In [0]:
from pyspark.sql import functions as F, Window

CAT = "workspace"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.silver")

DataFrame[]

In [0]:
df = spark.table(f"{CAT}.bronze.tb_movies_info")

# o bronze usa append, que aí o mesmo filme pode aparecer várias vezes.
df = df.withColumn("id", F.trim(F.col("id").cast("string"))).filter(F.col("id").isNotNull() & (F.col("id") != ""))

w = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

In [0]:
# normaliza
df = df.withColumn(
    "status_norm",
    F.trim(F.regexp_replace(
        F.regexp_replace(
            F.regexp_replace(F.lower(F.col("status")), r"[-_]+", " "),
            r"[^a-z ]", ""),
        r"\s+", " "))
)

# traduz
mapa_status = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado",
    "cancelled": "Cancelado",
}
status_pt = F.lit("Não Informado")
for k, v in mapa_status.items():
    status_pt = F.when(F.col("status_norm") == k, F.lit(v)).otherwise(status_pt)

df = df.withColumn("status_filme", status_pt)